# Latent Transforms for Template Similarity

## Why latent transforms?

`template_similarity()` compares density maps in raw screen space. But when
encoding and retrieval differ by a linear transform — different screen sizes,
calibration drift between sessions, or systematic participant differences —
raw correlation can underestimate true reinstatement.

Latent transforms address this by mapping both sides into a shared space
before computing similarity. You add them to `template_similarity()` with
two arguments; the core API stays unchanged.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from peyesim import (
    fixation_group, eye_density, template_similarity,
    coral_transform, latent_pca_transform, cca_transform,
)

## A concrete example

Suppose two devices record eye-movements on the same stimuli, but one device
scales coordinates differently. Let's simulate this:

In [ ]:
rng = np.random.default_rng(1)

def make_density(vec):
    fg = fixation_group(
        x=vec[:2] * 50 + 50,
        y=vec[2:4] * 50 + 50,
        onset=[0, 100],
        duration=[100, 100],
    )
    return eye_density(fg, sigma=30, xbounds=(0, 100), ybounds=(0, 100))

base_vecs = [rng.standard_normal(4) for _ in range(8)]

Now create a "source" set that is a scaled version of the reference — as if
recorded on a differently-calibrated device:

In [ ]:
scale_factors = np.array([2.0, 0.6, 1.7, 0.8])
source_vecs = [v * scale_factors for v in base_vecs]

ref_tab = pd.DataFrame({
    "id": list(range(len(base_vecs))),
    "density": [make_density(v) for v in base_vecs],
})
source_tab = pd.DataFrame({
    "id": list(range(len(source_vecs))),
    "density": [make_density(v) for v in source_vecs],
})

## Comparing raw vs. CORAL-transformed similarity

Without any transform, the scaling distortion reduces similarity:

In [ ]:
raw = template_similarity(
    ref_tab, source_tab,
    match_on="id",
    permutations=0,
    method="cosine",
)
print(f"Raw mean similarity: {raw['eye_sim'].mean():.3f}")

CORAL (CORrelation ALignment) whitens the source domain and re-colors it with
the reference covariance, correcting for the scaling difference:

In [ ]:
coral_res = template_similarity(
    ref_tab, source_tab,
    match_on="id",
    permutations=0,
    method="cosine",
    similarity_transform=coral_transform,
    similarity_transform_args={"comps": 4, "shrink": 1e-6},
)
print(f"CORAL mean similarity: {coral_res['eye_sim'].mean():.3f}")

In [ ]:
comparison = pd.DataFrame({
    "id": raw["id"],
    "Raw": raw["eye_sim"],
    "CORAL": coral_res["eye_sim"],
})

x = np.arange(len(comparison))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, comparison["Raw"], width, label="Raw", color="grey", alpha=0.8)
ax.bar(x + width/2, comparison["CORAL"], width, label="CORAL", color="steelblue", alpha=0.8)
ax.set_xlabel("Item")
ax.set_ylabel("Cosine Similarity")
ax.set_xticks(x)
ax.set_xticklabels(comparison["id"])
ax.legend()
plt.tight_layout()
plt.show()

CORAL recovers substantially higher similarity by correcting the covariance
mismatch between the two "devices."

## Available transforms

| Transform | Supervised? | Best for |
|:----------|:------------|:---------|
| `latent_pca_transform()` | No | Dimensionality reduction, mild noise smoothing |
| `coral_transform()` | No | Device/participant shifts (covariance-level differences) |
| `cca_transform()` | Yes (needs matched pairs) | Item-level alignment when pairings are reliable |

All are passed to `template_similarity()` via the `similarity_transform`
argument:

```python
template_similarity(
    ref_tab, source_tab,
    match_on="id",
    similarity_transform=cca_transform,
    similarity_transform_args={"comps": 10, "shrink": 0.01},
)
```

## Choosing a transform

- **PCA** is the safest default — dimensionality reduction with no assumptions
  about domain differences. Use when you want noise smoothing without domain
  adaptation.
- **CORAL** is unsupervised and assumes linear, covariance-level differences
  between domains. Good for calibration drift or different screen sizes.
- **CCA** is supervised and leverages matched pairs to find shared latent axes.
  Set `comps` modestly (5–15) and add `shrink` to stabilize small-N fits.

After the transform, `template_similarity()` still uses your chosen `method`
(cosine, pearson, fisherz, etc.) on the transformed vectors.

## Notes and limitations

- Multiscale densities are supported only when all scales share the same grid
  size; otherwise latent transforms will error.
- Regularization (`shrink`) and component count (`comps`) affect stability on
  small-N data — tune as needed.
- CORAL and CCA are linear; they will not correct nonlinear spatial warps.
  Keep `comps` low to reduce overfitting, especially with few pairs.
- See the [Overview notebook](01_overview.ipynb) for the core `template_similarity()` workflow
  without transforms.